# ChuckleNet: Train from Kaggle WavLM Embeddings

**Uses pre-extracted embeddings from Kaggle dataset**

- Dataset: `subhajitdas/chuckle-wavlm-prosody-utterance-extraction`
- 768-dim WavLM + 21-dim prosody
- ~175K utterances with labels
- Random 80/10/10 split

In [ ]:
# Install and authenticate
!pip install -q kaggle
!pip install -q transformers librosa scikit-learn

import os
os.environ['KAGGLE_USERNAME'] = 'dasrebel'
# Get from: https://www.kaggle.com/account → API → Create New Token
os.environ['KAGGLE_KEY'] = input('Enter your Kaggle API key: ')

from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

In [ ]:
# Download dataset from Kaggle
import os
DATA_DIR = '/content/gdrive/MyDrive/chuckle_kaggle_data'
os.makedirs(DATA_DIR, exist_ok=True)

print('📥 Downloading Kaggle dataset...')
!kaggle datasets download -d subhajitdas/chuckle-wavlm-prosody-utterance-extraction -p {DATA_DIR} --unzip
print(f'✅ Downloaded to {DATA_DIR}')

# List contents
!ls -la {DATA_DIR}

In [ ]:
# Load the data
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
import time

DATA_DIR = '/content/gdrive/MyDrive/chuckle_kaggle_data'

print('📂 Loading embeddings...')
data = []
with open(f'{DATA_DIR}/utterances_with_labels.json') as f:
    for line in f:
        data.append(json.loads(line))

print(f'✅ Loaded {len(data)} utterances')

# Analyze labels
labels = [d['label'] for d in data]
pos_count = sum(labels)
neg_count = len(labels) - pos_count
print(f'📊 Positive: {pos_count} ({pos_count/len(labels)*100:.1f}%)')
print(f'📊 Negative: {neg_count} ({neg_count/len(labels)*100:.1f}%)')

In [ ]:
# Random 80/10/10 split
import random
random.seed(42)
np.random.seed(42)

random.shuffle(data)

n = len(data)
val_size = int(n * 0.1)
test_size = int(n * 0.1)

train_data = data[val_size + test_size:]
val_data = data[:val_size]
test_data = data[val_size:val_size + test_size]

print(f'📊 Train: {len(train_data)} ({len(train_data)/n*100:.1f}%)')
print(f'📊 Val: {len(val_data)} ({len(val_data)/n*100:.1f}%)')
print(f'📊 Test: {len(test_data)} ({len(test_data)/n*100:.1f}%)')

# Verify split
for name, split in [('train', train_data), ('val', val_data), ('test', test_data)]:
    pos = sum(d['label'] for d in split)
    print(f'  {name}: {pos} positive ({pos/len(split)*100:.1f}%)')

In [ ]:
# Dataset class
class AudioTextDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        
        # WavLM embedding (768-dim)
        wavlm = torch.tensor(d['wavlm_embedding'], dtype=torch.float32)
        
        # Prosody features (21-dim)
        prosody = torch.tensor(d['prosody_features'], dtype=torch.float32)
        
        # Text embedding if available
        text_emb = torch.tensor(d.get('text_embedding', np.zeros(768)), dtype=torch.float32)
        
        # Combine: wavlm + prosody + text
        x = torch.cat([wavlm, prosody, text_emb])  # 768+21+768 = 1557
        
        y = torch.tensor(d['label'], dtype=torch.float32)
        
        return x, y

BATCH = 64

train_ds = AudioTextDataset(train_data)
val_ds = AudioTextDataset(val_data)
test_ds = AudioTextDataset(test_data)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH)
test_loader = DataLoader(test_ds, batch_size=BATCH)

print(f'✅ DataLoaders ready')
print(f'   Input dim: 1557 (768 WavLM + 21 prosody + 768 text)')

In [ ]:
# Model: Fusion MLP
class FusionModel(torch.nn.Module):
    def __init__(self, input_dim=1557):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 512),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(512, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

model = FusionModel().to(device)

# Class weights for imbalance
all_labels = [d['label'] for d in train_data]
pos_weight = torch.tensor([compute_class_weight('balanced', classes=[0,1], y=all_labels)[1]]).to(device)
print(f'⚖️ Pos weight: {pos_weight.item():.2f}')

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print('✅ Model ready!')

In [ ]:
# Training loop
EPOCHS = 10
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            val_preds.extend((out > 0.5).cpu().numpy())
            val_labels.extend(y.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        print(f'  ✅ New best! Saving...')
        torch.save(best_state, f'{DATA_DIR}/best_model.pt')

print(f'\n🏆 Best Val F1: {best_f1:.4f}')

In [ ]:
# Final evaluation on test set
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        test_preds.extend((out > 0.5).cpu().numpy())
        test_labels.extend(y.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'🏆 Test F1: {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))